# Concurrency & Error Handling

## What's covered

The final notebook in the language track. Two adjacent topics: how to run computations off the main thread, and how to handle the things that can go wrong — exceptions, missing resources, and the cleanup that has to happen either way.

**Concurrency:**

- The `Future[A]` mental model — a value that will (eventually) be available
- `ExecutionContext` — the thread pool every `Future` runs on, and why you need to pass one
- Creating futures with `Future { ... }`
- Combinators: `map`, `flatMap`, `filter`, `recover`, `recoverWith`, `fallbackTo`
- `for` comprehensions over futures — chaining without callback hell
- `Future.sequence` — turning `List[Future[A]]` into `Future[List[A]]`
- `Future.traverse` — `map` and `sequence` in one pass
- `Promise[A]` — the write side of a future, used at the boundary with callback-based ay-pee-eyes
- Blocking on a future — `Await.result`, when it's acceptable, when it's not
- Why callbacks are mostly avoided
- A short note on what comes after `Future` — Cats Effect, ZIO, and Scala 3 direct-style concurrency

**Error handling and resources:**

- `try` / `catch` / `finally` — the Java-style escape hatch
- Why `Try` and `Either` from notebook 06 are usually a better fit
- The `Using` resource-management construct — `try-with-resources` for Scala
- `Using.resource` for single resources, `Using` for multiple
- The `AutoCloseable` contract — what makes something `Using`-compatible
- Failing fast vs returning errors as data — the design call you make at every layer

## The `Future[A]` mental model

A `Future[A]` is a *placeholder for a value that will eventually be there*. It has two interesting states:

- **Incomplete** — the computation is still running.
- **Complete** — has either succeeded with an `A`, or failed with a `Throwable`.

Once a future completes, it never changes again. Multiple readers see the same result.

The contract: you don't ask *when* the value will be there. You schedule what to do *with it* when it arrives. The combinators (`map`, `flatMap`, `recover`) wrap that scheduling so it looks like ordinary chaining.

## `ExecutionContext` — the thread pool that runs futures

Every `Future` operation needs an `ExecutionContext`, which is a thread pool. The combinators don't run on the thread that called them — they run on the execution context.

You provide one via Scala's context-parameter machinery (notebook 08). Three common sources:

- **`scala.concurrent.ExecutionContext.global`** — the standard implicit, a `ForkJoinPool` sized to the number of CPU cores. Good default for general use.
- **A custom `Executor`-backed context** — when you need a different sizing strategy (smaller pool, bounded queue, named threads for debugging).
- **A library-specific context** — Akka actors come with their own; Spark internally uses its own scheduler.

The implicit-parameter shape is what every `Future` ay-pee-eye uses:

```scala
def expensive(x: Int)(using ec: ExecutionContext): Future[Int] =
  Future { x * x }
```

Callers either have an `ExecutionContext` in scope (most common: `import scala.concurrent.ExecutionContext.Implicits.global`), or pass one explicitly with `using`.

In [ ]:
import scala.concurrent.{Future, ExecutionContext}
import scala.concurrent.duration.*
import scala.concurrent.Await

// Bring the default ExecutionContext into implicit scope
given ExecutionContext = ExecutionContext.global

// Future { body } — schedules `body` on the EC, returns a Future immediately.
val f: Future[Int] = Future {
  println(s"  computing on ${Thread.currentThread.getName}")
  Thread.sleep(50)
  42
}

println("scheduled, not done yet")
val result = Await.result(f, 1.second)
println(s"result = $result")

## Future combinators

The combinators on `Future` mirror those on `Option`, `Try`, and `Either` from notebook 06. The pattern is the same — wrap an effect that might succeed or fail, chain operations that automatically propagate failure.

| Method | What it does |
|---|---|
| `map(f)` | apply `f` to the successful value, in the future's context |
| `flatMap(f)` | like `map` but `f` returns a `Future` itself |
| `filter(p)` | keep the value if `p` is true; otherwise fail with `NoSuchElementException` |
| `recover(pf)` | if the future failed, apply a partial function to the exception |
| `recoverWith(pf)` | like `recover`, but the handler returns a `Future` |
| `fallbackTo(other)` | if `this` failed, return `other` |
| `transform(f)` | uniform transformation of `Try` -> `Try` |
| `onComplete(callback)` | fire a side effect when the future completes — last-resort escape hatch |

In [ ]:
import scala.concurrent.{Future, ExecutionContext}
import scala.concurrent.duration.*
import scala.concurrent.Await

given ExecutionContext = ExecutionContext.global

def lookup(id: Int): Future[String] = Future {
  Thread.sleep(20)
  if id < 0 then throw new RuntimeException(s"bad id $id")
  else s"user-$id"
}

// map — transform the success
val nameFuture: Future[String] = lookup(42).map(_.toUpperCase)
println(Await.result(nameFuture, 1.second))   // USER-42

// recover — turn failure into success
val safeFuture: Future[String] = lookup(-1).recover {
  case _: RuntimeException => "anonymous"
}
println(Await.result(safeFuture, 1.second))   // anonymous

// fallbackTo — try another future if this one fails
val fb = lookup(-1).fallbackTo(lookup(7))
println(Await.result(fb, 1.second))           // user-7

## `for` comprehensions over futures

This is the cleanest way to chain multiple futures. Each `name <- futureExpr` binds the success value of one future; the chain short-circuits on the first failure.

`for`-comprehensions on `Future` desugar into `flatMap` and `map` calls — exactly the same translation as `for` over `Option` (notebook 06). Each step in the chain runs as soon as its dependencies complete; independent steps that you want to run in *parallel* must be started *before* the `for`.

In [ ]:
import scala.concurrent.{Future, ExecutionContext}
import scala.concurrent.duration.*
import scala.concurrent.Await

given ExecutionContext = ExecutionContext.global

def fetchName(id: Int):  Future[String] = Future { Thread.sleep(50); s"name-$id" }
def fetchAge(id: Int):   Future[Int]    = Future { Thread.sleep(50); 25 + id }
def fetchCity(id: Int):  Future[String] = Future { Thread.sleep(50); "bengaluru" }

// SEQUENTIAL — each future starts only after the previous completes (~150ms total)
val sequential = for
  name <- fetchName(1)
  age  <- fetchAge(1)
  city <- fetchCity(1)
yield s"$name, $age, $city"

println(Await.result(sequential, 2.seconds))

// PARALLEL — start all three futures first, THEN combine in for
//   This runs the three in parallel (~50ms total).
val nameF = fetchName(1)
val ageF  = fetchAge(1)
val cityF = fetchCity(1)

val parallel = for
  name <- nameF
  age  <- ageF
  city <- cityF
yield s"$name, $age, $city"

println(Await.result(parallel, 2.seconds))

## `Future.sequence` and `Future.traverse`

A common shape: you have a list of inputs, you want to fire off a future for each one, and end up with a future of the resulting list.

- **`Future.traverse(xs)(f)`** — apply `f` to each element to make `Future[B]`, then aggregate into one `Future[List[B]]`. Equivalent to `Future.sequence(xs.map(f))`, but in one pass.
- **`Future.sequence(futures)`** — given a `List[Future[A]]`, return `Future[List[A]]`. Use when you already have the list of futures.

Both fail the resulting future if any of the input futures fail — exception-style. For "succeed with whatever made it," combine with `recover` on each future first.

In [ ]:
import scala.concurrent.{Future, ExecutionContext}
import scala.concurrent.duration.*
import scala.concurrent.Await

given ExecutionContext = ExecutionContext.global

def lookup(id: Int): Future[String] = Future {
  Thread.sleep(20)
  s"user-$id"
}

// traverse — start one future per input, collect results
val ids = List(1, 2, 3, 4, 5)
val all: Future[List[String]] = Future.traverse(ids)(lookup)
println(Await.result(all, 2.seconds))

// sequence — start with a list of futures, aggregate
val futures: List[Future[String]] = ids.map(lookup)
val combined: Future[List[String]] = Future.sequence(futures)
println(Await.result(combined, 2.seconds))

## `Promise[A]` — the write side of a future

A `Promise[A]` is the bridge between a future and a code path that completes it externally. The promise has a `.future` you hand to callers; internally, your code calls `.success(v)` or `.failure(t)` exactly once when the value is ready.

Most everyday code doesn't need `Promise`. You reach for it when you're integrating with a **callback-based** ay-pee-eye — something that calls you back when work is done, and you need to expose that as a `Future` to your downstream code.

In [ ]:
import scala.concurrent.{Future, Promise, ExecutionContext}
import scala.concurrent.duration.*
import scala.concurrent.Await

given ExecutionContext = ExecutionContext.global

// Imagine an old-style callback API
def fetchOldStyle(id: Int)(callback: Either[Throwable, String] => Unit): Unit =
  // Pretend this runs on some library's thread
  new Thread(() =>
    Thread.sleep(30)
    if id < 0 then callback(Left(new RuntimeException(s"bad id $id")))
    else        callback(Right(s"old-style-$id"))
  ).start()

// Wrap it as a Future using Promise
def fetchFuture(id: Int): Future[String] =
  val promise = Promise[String]()
  fetchOldStyle(id) {
    case Right(v) => promise.success(v)
    case Left(t)  => promise.failure(t)
  }
  promise.future

println(Await.result(fetchFuture(7), 1.second))    // old-style-7

## Blocking on a future

`Await.result(future, duration)` blocks the calling thread until the future completes, then returns the success value (or rethrows the failure). `Await.ready(future, duration)` is the same but returns the future itself.

**When `Await` is acceptable:**

- At the top of a `main` method, where the program would otherwise exit before the future completes.
- In tests, to assert on the result.
- In the REPL or notebook, to see the result.

**When `Await` is a bug:**

- In production server code that handles a request per thread — you've serialized what was supposed to be concurrent.
- Inside a `Future`'s body — at best you waste a thread; at worst you deadlock the execution context.

In modern code the answer is: don't `Await`. Use `for` comprehensions and let the result flow forward, or write your application in a framework (Cats Effect, ZIO, Akka) that integrates concurrency end to end.

## Why callbacks are mostly avoided

`Future` provides `onComplete(callback)` for fire-and-forget — schedule something to happen when the future completes, return immediately. This is the lowest-level escape hatch.

Idiomatic Scala uses callbacks sparingly. Reasons:

- **Order is hard.** Two `onComplete`s on the same future fire in undefined order.
- **Errors are hard.** An exception inside a callback is logged and dropped unless you explicitly handle it.
- **Composition is hard.** Callbacks don't return a value, so chaining them produces nesting.

`map`, `flatMap`, and `for` are almost always cleaner. Reach for `onComplete` only when you really want side effects with no downstream computation — for example, a logger that records that a request completed, with nothing else depending on the outcome.

## What comes after `Future`

`scala.concurrent.Future` is the *standard library* concurrency primitive. It has known weaknesses — eager execution (a future starts the moment you create it; you can't cancel or restart it), no resource safety, awkward interaction with structured concurrency. Production Scala increasingly reaches for one of three alternatives:

- **Cats Effect (IO)** — Typelevel's effect type. A `IO[A]` describes a computation without running it; you compose, then call `unsafeRunSync` once at the edge. Supports cancellation, resource safety, structured concurrency.
- **ZIO** — Effect type with an explicit error and environment channel — `ZIO[R, E, A]`. Same idea as `IO` but with built-in dependency injection and typed errors.
- **Scala 3 direct-style concurrency** — `scala.concurrent.async` (experimental as of 2026): structured concurrency, cancellation, exception propagation, all without `flatMap` chains. Looks like ordinary sequential code but runs concurrently.

You do not need any of these to start. `Future` will carry most working code for years. The point is: if you find yourself reaching for `Await`, threading callbacks, or wrestling with cancellation, look at one of the above before writing more `Future` plumbing.

## Error handling — `try` / `catch` / `finally`

Scala has java-style `try`/`catch`/`finally`, and the bodies are expressions. The shape:

```scala
val result =
  try riskyOperation()
  catch
    case e: SomeException => fallbackValue
  finally
    cleanup()
```

`try`/`catch` is the right tool when you're at the edge of your program (a `main`, an HTTP request handler) and need to translate an exception into a response. Inside your logic, the `Try`, `Option`, and `Either` types from notebook 06 are usually cleaner — they make failure part of the type, instead of an out-of-band channel.

In [ ]:
def parseQuantity(s: String): Int =
  try
    s.toInt
  catch
    case _: NumberFormatException =>
      println(s"  bad input: $s")
      0
  finally
    println("  cleanup done")

println(parseQuantity("42"))
println("---")
println(parseQuantity("xyz"))

## `Try` vs `try` — when each fits

The relationship:

- `try` / `catch` is the **control-flow form** — block-local handling, then proceed.
- `Try` (notebook 06) is the **data form** — wrap the result, chain combinators, return it.

A rough decision rule:

- The exception is something you want to **handle right here** → `try` / `catch`.
- The exception is something you want to **return to the caller, who decides** → `Try` or `Either`.
- The exception comes from a **java api at a boundary** → `Try { javaCall() }` to convert at the seam.
- You want to **chain multiple operations**, any of which may fail → `Try` or `Either` (or `Future`), with `for`.

## `Using` — resource management

Scala 2.13 added `scala.util.Using` for "open this resource, use it, close it whether or not anything blows up." This is the Scala equivalent of java's *try-with-resources* and Python's `with` statement.

The shape:

```scala
import scala.util.Using

val contents: Try[String] = Using(Source.fromFile("data.txt")) { source =>
  source.getLines().mkString("\n")
}
```

`Using` takes the resource expression, runs your block with the resource, and **always closes the resource** afterward — even if your block threw. The result is wrapped in a `Try`.

There are two related forms:

- **`Using(...)(...)`** — returns a `Try[A]`, propagates failures from both the body and the close.
- **`Using.resource(...)(...)`** — runs the body and lets exceptions propagate. Use when you'd rather have the exception than the `Try`.

In [ ]:
import scala.util.Using
import scala.io.Source

// Set up a small temp file we can read back
import java.nio.file.{Files, Path}
val tmp = Files.createTempFile("scala-track-", ".txt")
Files.writeString(tmp, "line1\nline2\nline3")

// Reading with Using — file is closed automatically
val contents = Using(Source.fromFile(tmp.toFile)) { src =>
  src.getLines().toList
}
println(contents)   // Success(List(line1, line2, line3))

// Using.resource — body's value, exceptions propagate
val joined = Using.resource(Source.fromFile(tmp.toFile)) { src =>
  src.getLines().mkString(" | ")
}
println(joined)

// Cleanup
Files.delete(tmp)

## The `AutoCloseable` contract — what makes something `Using`-compatible

`Using` works with anything that has an `AutoCloseable` shape — a `close(): Unit` method. The standard library defines the `Releasable` type class to formalize this, and most resource-like java types (`Source`, `InputStream`, `BufferedWriter`, JDBC `Connection`) implement `AutoCloseable` already.

If you write your own resource type, give it a `close()` method and Scala's `Using` machinery will pick it up via the default `Releasable[AutoCloseable]` instance.

In [ ]:
import scala.util.Using

class Connection(name: String) extends AutoCloseable:
  println(s"  opening $name")
  def query(sql: String): String = s"<$name> result of: $sql"
  def close(): Unit = println(s"  closing $name")

val out = Using(Connection("db1")) { conn =>
  conn.query("SELECT * FROM users")
}
println(out)

// Using with multiple resources — declared with .apply on Using's helper
val combined = Using.Manager { use =>
  val a = use(Connection("db1"))
  val b = use(Connection("db2"))
  s"${a.query("a")} + ${b.query("b")}"
}
println(combined)
// Both connections are closed, in reverse order of acquisition

## Failing fast vs returning errors as data

A design call you make at every layer of every system. There is no universal answer; the rule of thumb:

- **At the edges** (your `main`, an HTTP handler, a CLI), exceptions that you can't sensibly handle should propagate to the top and produce a clean error response. Failing fast is fine.
- **In the middle** (business logic, library code), prefer `Either` or `Try` so callers can compose decisions about what to do. An exception thrown two layers down with no type information is harder to handle gracefully.
- **At java boundaries**, wrap with `Try { javaCall() }` to convert the exception into a value at the seam — then you're in the data world from then on.

When in doubt: a library that returns `Either[YourErrorType, A]` is easier to use *and* easier to evolve than one that throws — the error type is documented, exhaustively-matched, and stable across versions.

## What's next

You now have the full standard-library concurrency story — `Future`, `Promise`, `ExecutionContext`, `Await` — plus the error-handling and resource-management shapes that round out everyday Scala work.

This is also the end of the language track. The next steps for this material are applied — Spark in Scala (the `apache-spark/` repo bridges directly, with the same case-class, pattern-matching, and type-class shapes you have learned here). And the broader functional-effect ecosystem (Cats Effect, ZIO) if you want a deeper concurrency story than `Future` provides.

Bookmark this track and notebook 06 in particular — pattern matching, `Option`, `Try`, `Either`, and `for` comprehensions over them are the shapes that show up most often in everyday code.